# Which Words Carry the Decision

**Computational Text Analysis Pack · Module 07 — Classification**

*Technique:* Model coefficients as evidence; interpreting a linear classifier

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/06-supervised-classification/07c_which_words_carry_the_decision.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

For most humanities questions the accuracy is not the result. The **coefficients**
are.

A linear classifier assigns every word a weight per class. Reading those weights
tells you which vocabulary distinguishes the categories — a supervised
counterpart to the keyness of Module 04, with an important advantage: it accounts
for all words at once, so it does not double-count words that travel together.

## The tooling

**scikit-learn** end to end: `train_test_split`, `LogisticRegression`,
`LinearSVC`, `MultinomialNB`, `cross_val_score`. This is the library's core
competence and the reason it is the general practitioner of the Python text
stack.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

sub = df[df["language"] == "English"]
keep = sub["category"].value_counts()
sub = sub[sub["category"].isin(keep[keep >= 40].index)].reset_index(drop=True)

Xtr_txt, Xte_txt, ytr, yte = train_test_split(
    sub["text"], sub["category"], test_size=0.25, random_state=0,
    stratify=sub["category"])
vec = TfidfVectorizer(token_pattern=r"[\w\u0300-\u036f]+", min_df=3, max_features=5000,
                      sublinear_tf=True)
Xtr, Xte = vec.fit_transform(Xtr_txt), vec.transform(Xte_txt)
clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
vocab = np.array(vec.get_feature_names_out())
print(f"accuracy {accuracy_score(yte, clf.predict(Xte)):.3f} — "
      "good enough that the coefficients are worth reading")

In [ ]:
TOP = 12
for i, cls in enumerate(clf.classes_):
    coefs = clf.coef_[i]
    pos = np.argsort(coefs)[::-1][:TOP]
    neg = np.argsort(coefs)[:TOP]
    print(f"\n{cls.upper()}")
    print(f"  towards : {', '.join(vocab[pos])}")
    print(f"  against : {', '.join(vocab[neg])}")

In [ ]:
CLASS = clf.classes_[0]     # ← change this
i = list(clf.classes_).index(CLASS)
coefs = clf.coef_[i]
idx = np.argsort(coefs)
sel = np.concatenate([idx[:12], idx[-12:]])

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(vocab[sel], coefs[sel],
        color=["steelblue" if c < 0 else "indianred" for c in coefs[sel]])
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("coefficient")
ax.set_title(f"Words pushing towards and away from '{CLASS}'")
plt.tight_layout(); plt.show()

## Coefficients are not frequencies

Three warnings, all of which are ways published work goes wrong here.

1. **A large coefficient on a rare word is unstable.** Check the word's document
   frequency before quoting it.
2. **Correlated words split their weight.** If *election* and *elections* both
   appear, each gets roughly half the credit, and both look weaker than the
   concept is.
3. **The weight is relative to the other classes**, not absolute. A word can be
   common in a class and still have a small coefficient because it is common
   everywhere.

In [ ]:
# Guard against warning 1: how much evidence is behind each coefficient?
doc_freq = np.asarray((Xtr > 0).sum(axis=0)).ravel()
top_idx = np.argsort(clf.coef_[i])[::-1][:15]
print(f"Top words for '{CLASS}', with the evidence behind them:")
print(f"{'word':<18}{'coefficient':>13}{'train docs':>12}")
print("-" * 43)
for j in top_idx:
    flag = "  ← thin" if doc_freq[j] < 15 else ""
    print(f"{vocab[j]:<18}{clf.coef_[i][j]:>13.3f}{doc_freq[j]:>12}{flag}")

In [ ]:
# Compare against unsupervised keyness on the same data. Where they agree,
# the finding is robust to method; where they disagree, ask why.
from collections import Counter
def freqs(texts):
    c = Counter()
    for t in texts:
        c.update(tokenise(t))
    return c

target = freqs(sub[sub["category"] == CLASS]["text"])
rest = freqs(sub[sub["category"] != CLASS]["text"])
na, nb = sum(target.values()), sum(rest.values())
key = sorted(((w, (target[w] / na) / max(rest.get(w, 0) / nb, 1e-9), target[w])
              for w in target if target[w] >= 25),
             key=lambda r: -r[1])[:15]

model_words = set(vocab[np.argsort(clf.coef_[i])[::-1][:25]])
print(f"{'keyness word':<18}{'rate ratio':>11}{'  also top-25 in the model?'}")
print("-" * 58)
for w, ratio, n in key:
    print(f"{w:<18}{ratio:>11.1f}   {'yes' if w in model_words else 'no'}")

### 🔧 Your turn 1

Change `CLASS` and read the coefficient list for a category you know something
about.

Is there a word you did not expect? Run KWIC on it (05a). Either you learn
something about the corpus, or you find an artefact — a boilerplate phrase, a
section header — and remove it.

### 🔧 Your turn 2

Compare the model's top words with the keyness list in the last cell.

Words that appear in both are the ones to write about. A word that appears only
in the keyness list is often riding along with another word; a word only the
model favours is often doing work in combination with others.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Section headers, agency names and outlet boilerplate are the
usual surprises. They are perfectly good predictors and terrible evidence: the
model has learned to recognise the publication rather than the subject. Finding
them is a reason to clean and re-run, and to say in your methods that you did.

**Your turn 2.** Overlap between the two lists is typically substantial but far
from complete, and the disagreements are the interesting part. Keyness treats
each word independently, so it happily lists five near-synonyms; the model
distributes weight among them and may rank all five lower. Reporting the overlap
— "these eight terms are distinctive under both an unsupervised keyness test and
a supervised classifier" — is a much stronger claim than either method alone.

</details>

## Where this stops

You can use a classifier as an instrument for reading rather than a scoreboard.

Next: **Module 08 — topic modelling**, which finds themes with no labels at all,
and requires more scepticism than any other method in this pack.